# 1. Sampling Analysis

In [5]:
# sampling analysis

from pathlib import Path
import pandas as pd

# =============================================================================
# Configuration
# =============================================================================

BASE_DIR = Path.cwd().parents[1]

PARQUET_DIR = BASE_DIR / "processing" / "parquet_aligned"

files = sorted(PARQUET_DIR.glob("*.parquet"))

In [6]:
# =============================================================================
# Sampling Analysis
# =============================================================================

results = []

for file in files:

    device = file.stem.upper()

    df = pd.read_parquet(file)

    diffs = (
        df.index
        .to_series()
        .diff()
        .dt.total_seconds()
        .dropna()
    )

    results.append({

        "Device": device,

        "Rows": len(df),

        "Start": df.index.min(),
        "End": df.index.max(),

        "Median Interval (s)": diffs.median(),
        "Min Interval (s)": diffs.min(),
        "Max Interval (s)": diffs.max(),

        "Duplicate Timestamps": df.index.duplicated().sum(),

        "Backwards Timestamps": (diffs < 0).sum(),

        "Gaps > 5 min": (diffs > 300).sum(),
        "Gaps > 1 hour": (diffs > 3600).sum(),
        "Gaps > 1 day": (diffs > 86400).sum()

    })

summary = pd.DataFrame(results)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

display(summary)

,Device,Rows,Start,End,Median Interval (s),Min Interval (s),Max Interval (s),Duplicate Timestamps,Backwards Timestamps,Gaps > 5 min,Gaps > 1 hour,Gaps > 1 day
0,ADB,1570251,2018-08-09 15:49:22,2022-04-12 15:48:05,60.0,16.0,13330370.0,0,0,43,9,8
1,CRAC1,1570230,2018-08-09 15:49:22,2022-04-12 15:53:05,60.0,16.0,13330370.0,0,0,43,9,8
2,CRAC2,1570716,2018-08-09 15:49:22,2022-04-12 23:59:05,60.0,16.0,13330370.0,0,0,43,9,8
3,CRAC3,1570202,2018-08-09 15:49:22,2022-04-12 15:27:05,60.0,16.0,13330370.0,0,0,43,9,8
4,CRAC4,1570221,2018-08-09 15:49:22,2022-04-12 15:52:05,60.0,16.0,13330370.0,0,0,44,9,8
5,CRAC5,1570707,2018-08-09 15:49:22,2022-04-12 23:59:05,60.0,16.0,13330370.0,0,0,44,9,8
6,CRAC6,1570707,2018-08-09 15:49:22,2022-04-12 23:59:05,60.0,16.0,13330370.0,0,0,44,9,8
7,IUDB,1570220,2018-08-09 15:49:22,2022-04-12 15:43:05,60.0,16.0,13330370.0,0,0,43,9,8
8,NDB,1570241,2018-08-09 15:49:22,2022-04-12 15:53:05,60.0,16.0,13330370.0,0,0,43,9,8
9,UDB1,1570724,2018-08-09 15:49:22,2022-04-12 23:59:05,60.0,16.0,13330370.0,0,0,43,9,8


In [7]:
from pathlib import Path
import pandas as pd

# =============================================================================
# Long Sampling Gaps (> 1 day)
# =============================================================================

all_gaps = {}

for file in files:

    device = file.stem.upper()

    df = pd.read_parquet(file)

    diffs = (
        df.index
        .to_series()
        .diff()
        .dt.total_seconds()
        .dropna()
    )

    gaps = diffs[diffs > 86400]

    gap_records = []

    for end_time, duration_s in gaps.items():

        start_time = end_time - pd.Timedelta(seconds=duration_s)

        gap_records.append({

            "Gap Start": start_time,
            "Gap End": end_time,
            "Duration (days)": round(duration_s / 86400, 2)

        })

    all_gaps[device] = pd.DataFrame(gap_records)

# =============================================================================
# Print gaps
# =============================================================================

for device, gaps in all_gaps.items():

    print("\n" + "=" * 90)
    print(device)
    print("=" * 90)

    if gaps.empty:
        print("No gaps greater than one day.")
    else:
        display(gaps)


ADB


,Gap Start,Gap End,Duration (days)
0,2018-10-24 20:50:30,2018-11-02 11:18:56,8.60
1,2019-02-26 16:16:56,2019-03-22 08:57:01,23.69
2,2020-01-15 09:26:18,2020-01-16 15:44:32,1.26
3,2020-01-18 17:38:32,2020-01-27 09:23:15,8.66
4,2020-03-07 18:55:15,2020-04-22 14:48:47,45.83
5,2021-02-16 03:54:47,2021-07-20 10:47:37,154.29
6,2021-08-06 17:40:45,2021-08-10 08:18:45,3.61
7,2021-08-29 02:55:52,2021-09-02 08:22:36,4.23



CRAC1


,Gap Start,Gap End,Duration (days)
0,2018-10-24 20:50:30,2018-11-02 11:18:56,8.60
1,2019-02-26 16:16:56,2019-03-22 08:57:01,23.69
2,2020-01-15 09:26:18,2020-01-16 15:44:32,1.26
3,2020-01-18 17:38:32,2020-01-27 09:23:15,8.66
4,2020-03-07 18:55:15,2020-04-22 14:48:47,45.83
5,2021-02-16 03:54:47,2021-07-20 10:47:37,154.29
6,2021-08-06 17:40:45,2021-08-10 08:18:45,3.61
7,2021-08-29 02:55:52,2021-09-02 08:22:36,4.23



CRAC2


...

,Gap Start,Gap End,Duration (days)
0,2018-10-24 20:50:30,2018-11-02 11:18:56,8.60
1,2019-02-26 16:16:56,2019-03-22 08:57:01,23.69
2,2020-01-15 09:26:18,2020-01-16 15:44:32,1.26
3,2020-01-18 17:38:32,2020-01-27 09:23:15,8.66
4,2020-03-07 18:55:15,2020-04-22 14:48:47,45.83
5,2021-02-16 03:54:47,2021-07-20 10:47:37,154.29
6,2021-08-06 17:40:45,2021-08-10 08:18:45,3.61
7,2021-08-29 02:55:52,2021-09-02 08:22:36,4.23



ULC8


,Gap Start,Gap End,Duration (days)
0,2018-10-24 20:50:30,2018-11-02 11:18:56,8.60
1,2019-02-26 16:16:56,2019-03-22 08:57:01,23.69
2,2020-01-15 09:26:18,2020-01-16 15:44:32,1.26
3,2020-01-18 17:38:32,2020-01-27 09:23:15,8.66
4,2020-03-07 18:54:15,2020-04-22 14:48:47,45.83
5,2021-02-16 03:54:47,2021-07-20 10:47:37,154.29
6,2021-08-06 17:40:45,2021-08-10 08:18:45,3.61
7,2021-08-29 02:55:52,2021-09-02 08:22:36,4.23



ULC9


,Gap Start,Gap End,Duration (days)
0,2018-10-24 20:50:30,2018-11-02 11:18:56,8.60
1,2019-02-26 16:16:56,2019-03-22 08:57:01,23.69
2,2020-01-15 09:26:18,2020-01-16 15:44:32,1.26
3,2020-01-18 17:38:32,2020-01-27 09:23:15,8.66
4,2020-03-07 18:55:15,2020-04-22 14:48:47,45.83
5,2021-02-16 03:54:47,2021-07-20 10:47:37,154.29
6,2021-08-06 17:40:45,2021-08-10 08:18:45,3.61
7,2021-08-29 02:55:52,2021-09-02 08:22:36,4.23


In [8]:
# =============================================================================
# Gap Consistency Across Devices
# =============================================================================

reference = None

results = []

for device, gaps in all_gaps.items():

    starts = list(gaps["Gap Start"]) if not gaps.empty else []

    if reference is None:
        reference = starts

    results.append({

        "Device": device,
        "Number of Gaps": len(starts),
        "Matches Reference": starts == reference

    })

consistency = pd.DataFrame(results)

display(consistency)

,Device,Number of Gaps,Matches Reference
0,ADB,8,True
1,CRAC1,8,True
2,CRAC2,8,True
3,CRAC3,8,True
4,CRAC4,8,True
5,CRAC5,8,True
6,CRAC6,8,True
7,IUDB,8,True
8,NDB,8,True
9,UDB1,8,True
